In [ ]:
import numpy as np
from magtense.magstatics import Tiles, run_simulation, get_demag_tensor

np.random.seed(42)

In [ ]:
# 1. Setup
L = 240e-7 / 20  # 10cm cube
mu0 = 4 * np.pi * 1e-7
M_val = 1.2 / mu0
m_vec = np.array([0, 0, M_val])

In [ ]:
# Define "Distance" in cell sizes (e.g., 5x the cube size)
cell_multiplier = 30
R = L * cell_multiplier
num_points = 10

# 2. Generate Random Points with shape (N, 3)
# Drawing from a normal distribution ensures uniform distribution on a sphere
random_pts = np.random.normal(size=(num_points, 3))
# Normalize each row (axis=1) and scale to radius R
norms = np.linalg.norm(random_pts, axis=1, keepdims=True)
random_pts = (random_pts / norms) * R

In [ ]:
# 3. MagTense Simulation
# We convert the (N, 3) array to a (3, N) Fortran-ordered array for the solver
pts_for_sim = np.asarray(random_pts)

tiles = Tiles(n=1, M_rem=M_val, tile_type=[2], easy_axis=[0, 0, 1])
tiles.size = ([L, L, L], 0)
tiles.offset = ([0, 0, 0], 0)

_, H_pts = run_simulation(tiles, pts_for_sim)


In [ ]:


# 4. Analytical Comparison
def get_dipole_field(r_vec, m_vec):
    """Calculates the H-field of a dipole at vector r."""
    r_mag = np.linalg.norm(r_vec)
    # Dipole formula:
    # H = (1/4pi) * [ (3r(m.r)/r^5) - (m/r^3) ]
    r_hat = r_vec / r_mag

    #term1 = 3 * r_vec * np.dot(m_vec, r_vec) / (r_mag**5)
    #term2 = m_vec / (r_mag**3)


    term1 = 3 * r_hat * np.dot(m_vec, r_hat)
    term2 = m_vec
    return (1 / (4 * np.pi)) * (term1 - term2) / (r_mag**3)

print(f"Testing {num_points} random points at R = {cell_multiplier}L ({R}m)")
print(f"{'Point #':<10} | {'MagTense |H|':<15} | {'Dipole |H|':<15} | {'Error (%)':<10}")
print("-" * 65)
errors = []
for i in range(num_points):
    # h_magtense is already returned in (3, N) format from the solver
    h_mag = H_pts[i, :]
    # random_pts[i] gives us the (3,) vector for the i-th point
    h_dip = get_dipole_field(random_pts[i], m_vec * L**3)
    
    norm_m = np.linalg.norm(h_mag)
    norm_d = np.linalg.norm(h_dip)
    err = abs(norm_m - norm_d) / norm_m * 100
    errors.append(err)
    
    print(f"{i+1:<10} | {norm_m:<15.2e} | {norm_d:<15.2e} | {err:<10.2f}")

print("-" * 65)
print(f"Average Error: {np.mean(errors):.2f}%")

In [ ]:
demag_tensor_mag = get_demag_tensor(tiles, pts_for_sim)


In [ ]:
def calculate_field_from_tensor(K, M, volume=1.0):
    """
    Calculates the magnetic field H from a coupling/dipole tensor K.
    
    Parameters:
    - K: (3, 3) ndarray. The dipole/interaction tensor.
    - M: (3,) or (3, 1) ndarray. The magnetization vector (A/m).
    - volume: float. The volume of the cell (m^3). 
              Set to 1.0 if M is already the magnetic moment.
              
    Returns:
    - H: (3,) ndarray. The resulting magnetic field vector.
    """
    # Ensure M is a 1D array for clean multiplication
    M = np.asarray(M).flatten()
    
    # Magnetic Moment m = M * V
    m = M * volume
    
    # H = K @ m
    H = K @ m
    
    return H

In [ ]:
calculate_field_from_tensor(demag_tensor_mag[0,3], m_vec)

In [ ]:
H_pts

In [ ]:
def dipole_tensor_3x3(Rvec):
    """
    Computes the dipole field tensor: Kdip = (1/4πr³) * (3u uᵀ - I)
    Maps Magnetic Moment (m) to H-field (H = Kdip @ m)
    """
    r2 = np.sum(Rvec**2)
    rmag = np.sqrt(r2)
    
    if rmag < 1e-15:
        return np.zeros((3, 3))

    invR3 = 1.0 / (4 * np.pi * r2 * rmag)
    u = Rvec / rmag  # Normalized unit vector
    
    # np.outer(u, u) creates the 3x3 matrix u_i * u_j
    Kdip = (3.0 * np.outer(u, u) - np.eye(3)) * invR3
    return Kdip


demag_tensor_mag = get_demag_tensor(tiles, pts_for_sim)


print("\nComparing MagTense's Demag Tensor to Analytical Dipole Tensor:")
print(f"{'Point #':<10} | {'MagTense Tensor':<20} | {'Dipole Tensor':<20} | {'Error (%)':<10}")
print("-" * 70)
errors = []
for i in range(num_points):
    Rvec = random_pts[i]
    K_dip = dipole_tensor_3x3(Rvec) * L**3
    K_mag = demag_tensor_mag[0,i,:,:]

    print (f"Point {i+1}:")
    print("K_mag:")
    print(K_mag)
    print("K_dip:")
    print(K_dip)
    print(f"Difference (K_mag - K_dip):")
    print(K_mag - K_dip)
    print("-----------------------")
    
    # Compute Frobenius norm of the difference
    #error = np.linalg.norm(K_mag - K_dip, 'fro') / np.linalg.norm(K_dip, 'fro') * 100
    #errors.append(error)
    #
    #print(f"{i+1:<10} | {np.array2string(K_mag, precision=2, suppress_small=True):<20} | {np.array2string(K_dip, precision=2, suppress_small=True):<20} | {error:<10.2f}")